In [15]:
print("Day 4")

Day 4


In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score

In [17]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

for name, df in [("train.csv", train), ("test.csv", test)]:
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print("\nТипы колонок:")
    print(df.dtypes)
    print("\nПропуски:")
    print(df.isnull().sum())
    print()


=== train.csv ===
Shape: (891, 12)

Типы колонок:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Пропуски:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

=== test.csv ===
Shape: (418, 11)

Типы колонок:
PassengerId      int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Пропуски:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age 

In [18]:
# --- 4. HasCabin ---
for df in [train, test]:
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df.drop(columns=["Cabin"], inplace=True)

# --- 1. Age: медиана по Pclass + Sex (считаем только на train) ---
median_age = train.groupby(["Pclass", "Sex"])["Age"].median().to_dict()

def fill_age(df):
    def _fill(row):
        if pd.isna(row["Age"]):
            return median_age.get((row["Pclass"], row["Sex"]), df["Age"].median())
        return row["Age"]
    df["Age"] = df.apply(_fill, axis=1)

fill_age(train)
fill_age(test)

# --- 2. Embarked ---
train["Embarked"] = train["Embarked"].fillna("S")
test["Embarked"]  = test["Embarked"].fillna("S")

# --- 3. Fare ---
test["Fare"] = test["Fare"].fillna(test["Fare"].median())

# --- 5. Sex: male=0, female=1 ---
sex_map = {"male": 0, "female": 1}
train["Sex"] = train["Sex"].map(sex_map)
test["Sex"]  = test["Sex"].map(sex_map)

# --- 6. Embarked: S=0, C=1, Q=2 ---
embarked_map = {"S": 0, "C": 1, "Q": 2}
train["Embarked"] = train["Embarked"].map(embarked_map)
test["Embarked"]  = test["Embarked"].map(embarked_map)

# --- 7. FamilySize ---
for df in [train, test]:
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# --- 8. IsAlone ---
for df in [train, test]:
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# --- 9. Title ---
title_map = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}

def extract_title(df):
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
    df["Title"] = df["Title"].str.strip()
    df["Title"] = df["Title"].map(title_map).fillna(5).astype(int)

extract_title(train)
extract_title(test)

# --- 10. Удаляем лишние колонки ---
train.drop(columns=["Name", "Ticket", "PassengerId"], inplace=True)
test.drop(columns=["Name", "Ticket"], inplace=True)

print("=== train ===")
print(train.shape)
print(train.dtypes)
print(train.isnull().sum())
print()
print("=== test ===")
print(test.shape)
print(test.dtypes)
print(test.isnull().sum())


=== train ===
(891, 12)
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
HasCabin        int64
FamilySize      int64
IsAlone         int64
Title           int64
dtype: object
Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
HasCabin      0
FamilySize    0
IsAlone       0
Title         0
dtype: int64

=== test ===
(418, 12)
PassengerId      int64
Pclass           int64
Sex              int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked         int64
HasCabin         int64
FamilySize       int64
IsAlone          int64
Title            int64
dtype: object
PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
HasCabin       0
FamilySize     0
IsAlone  

In [19]:
features = ["Pclass", "Sex", "Age", "Fare", "Embarked",
            "FamilySize", "IsAlone", "HasCabin", "Title"]

X_train = train[features]
y_train = train["Survived"]
X_test  = test[features]

# --- 1. Обучение ---
model = LinearRegression()
model.fit(X_train, y_train)
train_preds = model.predict(X_train)

# --- 2. Подбор порога ---
best_threshold, best_acc = 0.5, 0.0
for threshold in np.arange(0.3, 0.71, 0.01):
    preds = (train_preds >= threshold).astype(int)
    acc = accuracy_score(y_train, preds)
    if acc > best_acc:
        best_acc = acc
        best_threshold = round(threshold, 2)

print(f"Лучший порог: {best_threshold:.2f}  |  Accuracy на train: {best_acc:.4f}")

# --- 3. Предсказания на test ---
test_preds = (model.predict(X_test) >= best_threshold).astype(int)

# --- 4. Сохранение submission.csv ---
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_preds
})
submission.to_csv("submission.csv", index=False)

print("\nПервые 5 строк submission.csv:")
print(submission.head())


Лучший порог: 0.62  |  Accuracy на train: 0.8249

Первые 5 строк submission.csv:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
